# 02 Build the analysis mart

Builds the shared analysis mart used by Q1 to Q4 from the cleaned CSVs produced by `cleaned_data/01_clean_data.ipynb`.

**Inputs** (`cleaned_data/`): `registration_fact.csv`, `coe_clean.csv`, `fleet_clean.csv`, `mvp02_clean.csv`

**Outputs** (`mart/`):

| File | Grain | Used by |
|---|---|---|
| `segment_month_full.csv` | month × segment (49 × 3 = 147 rows) | Q2 paired t-tests, Q3 monthly capture, Q4 COE tests |
| `segment_period.csv` | period × segment (4 × 3 = 12 rows) | Q1 shift-share, Q2 growth, Q3 Capture Gap |
| `competitor_gap_p3p4.csv` | make × segment, P3 to P4 | Q3 competitor benchmarking |
| `mart_check_log.csv` | one row per check | Data governance evidence |

**Definitions** (report Section 3.2): BEV = LTA fuel type `Electric`; scope = SUV, Sedan and MPV; segment market = BEV registrations of all makes in the segment; BYD share = BYD BEV registrations ÷ segment market. Periods run August to July: P1 = Aug 2022 to Jul 2023, through P4 = Aug 2025 to Jul 2026.

**Checks**: 7 input checks run before the build and 12 mart checks run after it. Any failed check stops the notebook before files are written.

In [1]:
import os
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

HERE = Path.cwd()

def find_dir(name: str) -> Path:
    """Locate a folder whether the notebook runs from mart/, the repo root, or a flat Colab folder."""
    for d in (HERE.parent / name, HERE / name, HERE):
        if (d / "registration_fact.csv").exists():
            return d
    raise FileNotFoundError("cleaned_data folder with registration_fact.csv not found")

CLEAN = find_dir("cleaned_data")
OUT = HERE if HERE.name == "mart" else HERE / "mart"
OUT.mkdir(exist_ok=True)

SEGMENTS = ["SUV", "Sedan", "MPV"]
BEV_FUEL = "Electric"
BYD = "BYD"
FIRST_MONTH, LAST_MONTH = "2022-07", "2026-07"
THIN_MONTH = 50                      # segment-months below 50 BEVs are flagged

check_log = []
def check(stage: str, name: str, passed: bool, detail):
    """Record a check; stop the run if it fails."""
    check_log.append({"stage": stage, "check": name, "result": "PASS" if passed else "FAIL", "detail": detail})
    print(f"  [{'PASS' if passed else 'FAIL'}] {stage}: {name} -> {detail}")
    if not passed:
        raise ValueError(f"{stage} check failed: {name} ({detail})")

def period_of(month: str):
    """Aug-Jul periods: 2022-08..2023-07 = P1, ..., 2025-08..2026-07 = P4; 2022-07 has no period."""
    y, m = int(month[:4]), int(month[5:7])
    k = (y - 2022) + (1 if m >= 8 else 0)
    return f"P{k}" if k >= 1 else None

print("Cleaned data folder:", os.path.relpath(CLEAN, HERE))
print("Output folder:", os.path.relpath(OUT, HERE))

Cleaned data folder: ../cleaned_data
Output folder: .


## 1. Load inputs

In [2]:
reg = pd.read_csv(CLEAN / "registration_fact.csv", encoding="utf-8-sig")
coe = pd.read_csv(CLEAN / "coe_clean.csv", encoding="utf-8-sig")
fleet = pd.read_csv(CLEAN / "fleet_clean.csv", encoding="utf-8-sig")
mvp02 = pd.read_csv(CLEAN / "mvp02_clean.csv", encoding="utf-8-sig")

coe["month"] = coe["month"].astype(str).str[:7]
fleet["month"] = fleet["month"].astype(str).str[:7]
expected_months = pd.period_range(FIRST_MONTH, LAST_MONTH, freq="M").strftime("%Y-%m").tolist()
print(f"registration_fact {len(reg):,} rows | coe {len(coe):,} | fleet {len(fleet):,} | mvp02 {len(mvp02):,}")

registration_fact 26,898 rows | coe 1,975 | fleet 3,611 | mvp02 6,896


## 2. Input checks (7)

In [3]:
key = ["month", "make", "fuel_type", "body"]

# I1 Fact table grain is unique
dups = int(reg.duplicated(key).sum())
check("input", "I1 fact keys unique (month x make x fuel x body)", dups == 0, f"{len(reg):,} rows, {dups} duplicate keys")

# I2 All 49 months present, Jul 2022 to Jul 2026
months = sorted(reg["month"].unique())
check("input", "I2 49 contiguous months, 2022-07 to 2026-07", months == expected_months, f"{len(months)} months, {months[0]} to {months[-1]}")

# I3 Registrations are complete, whole and non-negative
bad = int(reg["registrations"].isna().sum() + (reg["registrations"] < 0).sum() + (reg["registrations"] % 1 != 0).sum())
check("input", "I3 registrations complete, whole and >= 0", bad == 0, f"{bad} invalid values")

# I4 Volume carried over from cleaning
total = int(reg["registrations"].sum())
check("input", "I4 total registrations match the cleaning log", total == 172_394, f"{total:,}")

# I5 Grid: every month x make x fuel combination lists all six body types
grid = reg.groupby(["month", "make", "fuel_type"]).size()
check("input", "I5 every month-make-fuel combination has 6 body types", bool((grid == 6).all()), f"{len(grid):,} combinations x 6 = {len(grid) * 6:,} rows")

# I6 COE Category A and B available for every month
coe_ab = coe[coe["vehicle_class"].isin(["Category A", "Category B"]) & coe["month"].isin(expected_months)]
coe_cover = coe_ab.groupby("month")["vehicle_class"].nunique().reindex(expected_months)
check("input", "I6 COE Cat A and B present in all 49 months", bool((coe_cover == 2).all()), f"{int((coe_cover == 2).sum())} of 49 months")

# I7 Car population (total and Electric) available for every month
cars = fleet[(fleet["category"] == "Cars") & fleet["month"].isin(expected_months)]
fleet_cover = cars[cars["type"] == BEV_FUEL].groupby("month").size().reindex(expected_months)
check("input", "I7 car population incl. Electric present in all 49 months", bool(fleet_cover.notna().all()), f"{int(fleet_cover.notna().sum())} of 49 months")

  [PASS] input: I1 fact keys unique (month x make x fuel x body) -> 26,898 rows, 0 duplicate keys
  [PASS] input: I2 49 contiguous months, 2022-07 to 2026-07 -> 49 months, 2022-07 to 2026-07
  [PASS] input: I3 registrations complete, whole and >= 0 -> 0 invalid values
  [PASS] input: I4 total registrations match the cleaning log -> 172,394
  [PASS] input: I5 every month-make-fuel combination has 6 body types -> 4,483 combinations x 6 = 26,898 rows
  [PASS] input: I6 COE Cat A and B present in all 49 months -> 49 of 49 months
  [PASS] input: I7 car population incl. Electric present in all 49 months -> 49 of 49 months


## 3. Build `segment_month_full` (one row per month and segment)

In [4]:
reg["is_bev"] = reg["fuel_type"].eq(BEV_FUEL)
reg["is_byd"] = reg["make"].eq(BYD)
scope = reg[reg["body"].isin(SEGMENTS)].rename(columns={"body": "segment"})
bev = scope[scope["is_bev"]]
idx = ["month", "segment"]

base = (scope.groupby(idx)["registrations"].sum().rename("market_all").to_frame()
        .join(bev.groupby(idx)["registrations"].sum().rename("market_bev"))
        .join(bev[bev["is_byd"]].groupby(idx)["registrations"].sum().rename("byd_bev"))
        .join(scope[scope["is_byd"]].groupby(idx)["registrations"].sum().rename("byd_all"))
        .join(scope[scope["is_byd"] & scope["fuel_type"].str.contains("Plug-In")]
              .groupby(idx)["registrations"].sum().rename("byd_phev"))
        .join(bev[bev["registrations"] > 0].groupby(idx)["make"].nunique().rename("n_bev_makes"))
        .fillna(0).astype("int64").reset_index())

# Largest non-BYD BEV competitor in each segment-month
rivals = (bev[~bev["is_byd"] & (bev["registrations"] > 0)]
          .sort_values(["month", "segment", "registrations", "make"], ascending=[True, True, False, True])
          .drop_duplicates(idx)[idx + ["make", "registrations"]]
          .rename(columns={"make": "top_rival", "registrations": "top_rival_bev"}))
mart = base.merge(rivals, on=idx, how="left")
mart["top_rival_bev"] = mart["top_rival_bev"].fillna(0).astype("int64")

# Calendar fields
mart["year"] = mart["month"].str[:4].astype(int)
mart["month_num"] = mart["month"].str[5:7].astype(int)
mart["period"] = mart["month"].map(period_of)

# Shares and flags
share = lambda num, den: np.where(den > 0, 100 * num / den.where(den > 0), np.nan)
mart["byd_share_pct"] = share(mart["byd_bev"], mart["market_bev"])
mart["bev_penetration_pct"] = share(mart["market_bev"], mart["market_all"])
mart["top_rival_share_pct"] = share(mart["top_rival_bev"], mart["market_bev"])
mart["thin_month"] = (mart["market_bev"] < THIN_MONTH).astype(int)

# Same calendar month one year earlier (paired months) and monthly Capture Gap
mart = mart.sort_values(["segment", "month"]).reset_index(drop=True)
g = mart.groupby("segment")
mart["market_bev_prev_year"] = g["market_bev"].shift(12)
mart["byd_bev_prev_year"] = g["byd_bev"].shift(12)
mart["byd_share_prev_year_pct"] = g["byd_share_pct"].shift(12)
mart["benchmark_bev_month"] = mart["byd_share_prev_year_pct"] / 100 * mart["market_bev"]
mart["capture_gap_month"] = mart["byd_bev"] - mart["benchmark_bev_month"]

# Month-on-month changes for the Q4 tests
mart["market_bev_mom_pct"] = g["market_bev"].pct_change() * 100
mart["byd_share_mom_pp"] = g["byd_share_pct"].diff()

# COE: mean premium and summed quota and bids over the month's bidding exercises
coe_m = (coe_ab.assign(cat=coe_ab["vehicle_class"].str[-1].str.lower())
         .groupby(["month", "cat"]).agg(premium=("premium", "mean"), quota=("quota", "sum"),
                                         bids_received=("bids_received", "sum"), bids_success=("bids_success", "sum"))
         .unstack("cat"))
coe_m.columns = [f"cat_{c}_{v}" for v, c in coe_m.columns]
coe_m = coe_m.reset_index().sort_values("month")
for col in ["cat_a_premium", "cat_b_premium", "cat_a_quota", "cat_b_quota"]:
    coe_m[f"{col}_mom_pct"] = coe_m[col].pct_change() * 100
coe_m["n_exercises"] = coe_ab.groupby("month")["bidding_no"].nunique().reindex(coe_m["month"]).to_numpy()
mart = mart.merge(coe_m, on="month", how="left")

# Car population context
fleet_m = (cars.groupby("month")["number"].sum().rename("cars_total").to_frame()
           .join(cars[cars["type"] == BEV_FUEL].groupby("month")["number"].sum().rename("cars_bev")).reset_index())
fleet_m["fleet_bev_share_pct"] = 100 * fleet_m["cars_bev"] / fleet_m["cars_total"]
mart = mart.merge(fleet_m, on="month", how="left")

ordered = ["month", "year", "month_num", "period", "segment",
           "market_all", "market_bev", "bev_penetration_pct", "n_bev_makes", "thin_month",
           "byd_bev", "byd_share_pct", "byd_all", "byd_phev",
           "top_rival", "top_rival_bev", "top_rival_share_pct",
           "market_bev_prev_year", "byd_bev_prev_year", "byd_share_prev_year_pct",
           "benchmark_bev_month", "capture_gap_month",
           "market_bev_mom_pct", "byd_share_mom_pp",
           "cat_a_premium", "cat_b_premium", "cat_a_quota", "cat_b_quota",
           "cat_a_bids_received", "cat_b_bids_received", "cat_a_bids_success", "cat_b_bids_success",
           "n_exercises", "cat_a_premium_mom_pct", "cat_b_premium_mom_pct",
           "cat_a_quota_mom_pct", "cat_b_quota_mom_pct",
           "cars_total", "cars_bev", "fleet_bev_share_pct"]
segment_month_full = mart[ordered].sort_values(["month", "segment"]).reset_index(drop=True)
print(f"segment_month_full: {len(segment_month_full)} rows x {segment_month_full.shape[1]} fields")
segment_month_full.head()

segment_month_full: 147 rows x 40 fields


,month,year,month_num,period,segment,market_all,market_bev,bev_penetration_pct,n_bev_makes,thin_month,...,cat_a_bids_success,cat_b_bids_success,n_exercises,cat_a_premium_mom_pct,cat_b_premium_mom_pct,cat_a_quota_mom_pct,cat_b_quota_mom_pct,cars_total,cars_bev,fleet_bev_share_pct
0,2022-07,2022,7,NaN,MPV,397,93,23.425693,2,0,...,1226,1048,2,NaN,NaN,NaN,NaN,648729,4716,0.72696
1,2022-07,2022,7,NaN,SUV,981,108,11.009174,8,0,...,1226,1048,2,NaN,NaN,NaN,NaN,648729,4716,0.72696
2,2022-07,2022,7,NaN,Sedan,729,31,4.252401,5,1,...,1226,1048,2,NaN,NaN,NaN,NaN,648729,4716,0.72696
3,2022-08,2022,8,P1,MPV,297,43,14.478114,2,1,...,1092,944,2,7.003824,0.550497,-11.24498,-10.341556,649165,4990,0.76868
4,2022-08,2022,8,P1,SUV,875,100,11.428571,11,0,...,1092,944,2,7.003824,0.550497,-11.24498,-10.341556,649165,4990,0.76868


## 4. Build `segment_period` and `competitor_gap_p3p4`

In [5]:
in_period = bev[bev["month"].map(period_of).notna()].assign(period=lambda d: d["month"].map(period_of))
pidx = ["period", "segment"]
sp = (in_period.groupby(pidx)["registrations"].sum().rename("market_bev").to_frame()
      .join(in_period[in_period["is_byd"]].groupby(pidx)["registrations"].sum().rename("byd_bev"))
      .join(scope[scope["month"].map(period_of).notna()].assign(period=lambda d: d["month"].map(period_of))
            .groupby(pidx)["registrations"].sum().rename("market_all"))
      .fillna(0).astype("int64").reset_index().sort_values(["segment", "period"]))
sp["byd_share"] = sp["byd_bev"] / sp["market_bev"]                       # proportion
sp["bev_penetration_pct"] = 100 * sp["market_bev"] / sp["market_all"]
g = sp.groupby("segment")
M0, M1 = g["market_bev"].shift(1), sp["market_bev"]
s0, s1 = g["byd_share"].shift(1), sp["byd_share"]
B0 = g["byd_bev"].shift(1)
sp["market_growth_pct"] = 100 * (M1 / M0 - 1)
sp["byd_growth_pct"] = 100 * (sp["byd_bev"] / B0 - 1)
sp["share_change_pp"] = 100 * (s1 - s0)
sp["benchmark_bev"] = s0 * M1                                             # hold previous-period share
sp["capture_gap"] = sp["byd_bev"] - sp["benchmark_bev"]
sp["market_effect"] = (s0 + s1) / 2 * (M1 - M0)                          # shift-share, Appendix E.1
sp["share_effect"] = (M0 + M1) / 2 * (s1 - s0)
sp["interaction"] = (s1 - s0) * (M1 - M0)
sp["byd_change"] = sp["byd_bev"] - B0
sp["byd_share_pct"] = 100 * sp["byd_share"]
segment_period = sp.drop(columns="byd_share").sort_values(["period", "segment"]).reset_index(drop=True)

# Competitors: each make's own P3 share held into P4
cp = (bev[bev["month"].map(period_of).isin(["P3", "P4"])]
      .assign(period=lambda d: d["month"].map(period_of))
      .groupby(["segment", "make", "period"])["registrations"].sum().unstack("period", fill_value=0).reset_index())
mkt = segment_period.set_index(["period", "segment"])["market_bev"]
cp["p3_share_pct"] = 100 * cp["P3"] / cp["segment"].map(mkt.xs("P3"))
cp["p4_share_pct"] = 100 * cp["P4"] / cp["segment"].map(mkt.xs("P4"))
cp["share_change_pp"] = cp["p4_share_pct"] - cp["p3_share_pct"]
cp["benchmark_bev"] = cp["p3_share_pct"] / 100 * cp["segment"].map(mkt.xs("P4"))
cp["capture_gap"] = cp["P4"] - cp["benchmark_bev"]
competitor_gap_p3p4 = (cp.rename(columns={"P3": "p3_bev", "P4": "p4_bev"})
                       .sort_values(["segment", "capture_gap"], ascending=[True, False]).reset_index(drop=True))
print(f"segment_period: {len(segment_period)} rows | competitor_gap_p3p4: {len(competitor_gap_p3p4)} rows")
segment_period[segment_period["period"] == "P4"][["segment", "market_bev", "byd_bev", "byd_share_pct", "share_change_pp", "capture_gap"]].round(2)

segment_period: 12 rows | competitor_gap_p3p4: 144 rows


,segment,market_bev,byd_bev,byd_share_pct,share_change_pp,capture_gap
9,MPV,3691,2709,73.39,-12.87,-474.94
10,SUV,22794,7602,33.35,-6.26,-1426.12
11,Sedan,4510,2358,52.28,21.71,979.22


## 5. Mart checks (12)

In [6]:
smf = segment_month_full

# M1 Grain: one row per month and segment
check("mart", "M1 147 rows, unique month x segment", len(smf) == 147 and not smf.duplicated(["month", "segment"]).any(),
      f"{len(smf)} rows")

# M2 Core fields complete
core = ["market_all", "market_bev", "byd_bev", "byd_share_pct", "bev_penetration_pct", "cat_a_premium",
        "cat_b_premium", "cat_a_quota", "cat_b_quota", "cars_total", "cars_bev"]
nulls = int(smf[core].isna().sum().sum())
check("mart", "M2 no missing values in core fields", nulls == 0, f"{nulls} missing")

# M3 Logical bounds: BYD BEV <= market BEV <= all-fuel market
bounds = int(((smf["byd_bev"] > smf["market_bev"]) | (smf["market_bev"] > smf["market_all"])).sum())
check("mart", "M3 byd_bev <= market_bev <= market_all", bounds == 0, f"{bounds} violations")

# M4 Shares lie between 0 and 100
pct = ["byd_share_pct", "bev_penetration_pct", "top_rival_share_pct", "fleet_bev_share_pct"]
out_of_range = int(((smf[pct] < 0) | (smf[pct] > 100)).sum().sum())
check("mart", "M4 all shares within 0 to 100", out_of_range == 0, f"{out_of_range} out of range")

# M5 Independent recomputation with SQL (same logic as view v_segment_month in SQL/Group20_BYD_BEV_schema_1.sql)
con = sqlite3.connect(":memory:")
reg[["month", "make", "fuel_type", "body", "registrations"]].to_sql("fact_registration", con, index=False)
sql = pd.read_sql_query("""
    SELECT month, body AS segment,
           SUM(registrations) AS market_all,
           SUM(CASE WHEN fuel_type = 'Electric' THEN registrations ELSE 0 END) AS market_bev,
           SUM(CASE WHEN fuel_type = 'Electric' AND make = 'BYD' THEN registrations ELSE 0 END) AS byd_bev
    FROM fact_registration
    WHERE body IN ('SUV', 'Sedan', 'MPV')
    GROUP BY month, body""", con)
con.close()
cmp_ = smf.merge(sql, on=["month", "segment"], suffixes=("", "_sql"))
diff = int(sum((cmp_[c] - cmp_[f"{c}_sql"]).abs().sum() for c in ["market_all", "market_bev", "byd_bev"]))
check("mart", "M5 matches independent SQL recomputation", len(cmp_) == 147 and diff == 0, f"total absolute difference {diff}")

# M6 Mart totals reconcile to the fact table
fact_bev = int(reg.loc[reg["is_bev"] & reg["body"].isin(SEGMENTS), "registrations"].sum())
check("mart", "M6 in-scope BEV total equals fact table", int(smf["market_bev"].sum()) == fact_bev, f"{fact_bev:,} BEVs")

# M7 Period table equals the sum of its monthly rows
roll = smf.dropna(subset=["period"]).groupby(["period", "segment"])[["market_bev", "byd_bev"]].sum()
sp_ = segment_period.set_index(["period", "segment"])[["market_bev", "byd_bev"]]
check("mart", "M7 12 period rows reconcile to monthly rows", len(sp_) == 12 and roll.equals(sp_.loc[roll.index]), f"{len(sp_)} rows")

# M8 Shift-share identity: market effect + share effect = BYD change
t = segment_period.dropna(subset=["market_effect"])
err = float((t["market_effect"] + t["share_effect"] - t["byd_change"]).abs().max())
check("mart", "M8 market + share effect = BYD change", err < 1e-6, f"max error {err:.2e} over {len(t)} transitions")

# M9 Capture Gap = share effect + half the interaction term (report Section 8.4.4)
err = float((t["share_effect"] + t["interaction"] / 2 - t["capture_gap"]).abs().max())
check("mart", "M9 Capture Gap = share effect + interaction / 2", err < 1e-6, f"max error {err:.2e}")

# M10 COE months align one to one with registration months
coe_months = sorted(smf.dropna(subset=["cat_b_quota"])["month"].unique())
check("mart", "M10 COE months align with registration months", coe_months == expected_months and bool((smf["n_exercises"] >= 1).all()),
      f"{len(coe_months)} months")

# M11 Monthly registrations reconcile with MVP02 annual totals
recon = {y: (int(reg.loc[reg["month"].str[:4] == str(y), "registrations"].sum()), int(mvp02.loc[mvp02["year"] == y, "number"].sum()))
         for y in (2023, 2024, 2025)}
check("mart", "M11 monthly = MVP02 annual totals, 2023-2025", all(a == b for a, b in recon.values()),
      "; ".join(f"{y}: {a:,} vs {b:,}" for y, (a, b) in recon.items()))

# M12 Frozen headline figures reproduce (report, frozen 1 Oct)
p4 = segment_period[segment_period["period"] == "P4"].set_index("segment")
tot = segment_period.groupby("period")[["byd_bev", "market_bev"]].sum()
headline = {
    "SUV Capture Gap": (round(p4.loc["SUV", "capture_gap"]), -1426),
    "Sedan Capture Gap": (round(p4.loc["Sedan", "capture_gap"]), 979),
    "MPV Capture Gap": (round(p4.loc["MPV", "capture_gap"]), -475),
    "BYD P3 share %": (round(100 * tot.loc["P3", "byd_bev"] / tot.loc["P3", "market_bev"], 1), 46.6),
    "BYD P4 share %": (round(100 * tot.loc["P4", "byd_bev"] / tot.loc["P4", "market_bev"], 1), 40.9),
    "BYD change P3 to P4": (int(tot.loc["P4", "byd_bev"] - tot.loc["P3", "byd_bev"]), 4147),
}
check("mart", "M12 frozen headline figures reproduce", all(a == b for a, b in headline.values()),
      "; ".join(f"{k} {a}" for k, (a, b) in headline.items()))

  [PASS] mart: M1 147 rows, unique month x segment -> 147 rows
  [PASS] mart: M2 no missing values in core fields -> 0 missing
  [PASS] mart: M3 byd_bev <= market_bev <= market_all -> 0 violations
  [PASS] mart: M4 all shares within 0 to 100 -> 0 out of range


  [PASS] mart: M5 matches independent SQL recomputation -> total absolute difference 0
  [PASS] mart: M6 in-scope BEV total equals fact table -> 63,131 BEVs
  [PASS] mart: M7 12 period rows reconcile to monthly rows -> 12 rows
  [PASS] mart: M8 market + share effect = BYD change -> max error 4.55e-13 over 9 transitions
  [PASS] mart: M9 Capture Gap = share effect + interaction / 2 -> max error 2.27e-13
  [PASS] mart: M10 COE months align with registration months -> 49 months
  [PASS] mart: M11 monthly = MVP02 annual totals, 2023-2025 -> 2023: 30,225 vs 30,225; 2024: 43,022 vs 43,022; 2025: 52,678 vs 52,678
  [PASS] mart: M12 frozen headline figures reproduce -> SUV Capture Gap -1426; Sedan Capture Gap 979; MPV Capture Gap -475; BYD P3 share % 46.6; BYD P4 share % 40.9; BYD change P3 to P4 4147


## 6. Export

In [7]:
segment_month_full.to_csv(OUT / "segment_month_full.csv", index=False, encoding="utf-8-sig")
segment_period.to_csv(OUT / "segment_period.csv", index=False, encoding="utf-8-sig")
competitor_gap_p3p4.to_csv(OUT / "competitor_gap_p3p4.csv", index=False, encoding="utf-8-sig")
log_df = pd.DataFrame(check_log)
log_df.to_csv(OUT / "mart_check_log.csv", index=False, encoding="utf-8-sig")

print(f"Checks passed: {int((log_df['result'] == 'PASS').sum())} of {len(log_df)} "
      f"({int((log_df['stage'] == 'input').sum())} input, {int((log_df['stage'] == 'mart').sum())} mart)")
print(f"segment_month_full.csv  {segment_month_full.shape[0]} rows x {segment_month_full.shape[1]} fields")
print(f"segment_period.csv      {segment_period.shape[0]} rows x {segment_period.shape[1]} fields")
print(f"competitor_gap_p3p4.csv {competitor_gap_p3p4.shape[0]} rows x {competitor_gap_p3p4.shape[1]} fields")

Checks passed: 19 of 19 (7 input, 12 mart)
segment_month_full.csv  147 rows x 40 fields
segment_period.csv      12 rows x 16 fields
competitor_gap_p3p4.csv 144 rows x 9 fields
